In [3]:
"""
Évaluation des modèles OpenRouter sur le classeur QCM (4 options A/B/C/D)
--------------------------------------------------------------------------------
Colonnes attendues dans la feuille :
  item_id, item_type, Axe, Difficulty, intent_group, intent_category,
  input, response_A, response_B, response_C, response_D, original_key, final_answer

Le script ajoute pour chaque modèle :
  response_<nom_court>_raw    -> texte brut renvoyé par le modèle
  response_<nom_court>        -> lettre parsée ("A".."D" ou "UNPARSED")

Caractéristiques :
- Appels concurrents (MAX_WORKERS, 10 par défaut).
- Gestion automatique du champ "reasoning" : on tente d'abord désactivé ; si
  l'endpoint répond « Reasoning is mandatory... », le modèle bascule en reasoning
  activé (puis, en dernier recours, le champ est omis). Le mode qui marche est
  mémorisé pour tous les appels suivants de ce modèle.
- Sauvegarde régulière + reprise : les lignes déjà remplies sont sautées, donc il
  suffit de ré-exécuter la cellule après un crash ou un Ctrl-C.
- Aucune erreur réseau/API/parsing n'arrête le run.
- Récapitulatif d'exactitude en fin de run (colonne final_answer).

À faire avant de lancer :
1. Mets ta clé dans OPENROUTER_API_KEY (variable d'environnement ou en dur).
2. Vérifie EXCEL_PATH et SHEET_NAME.
"""

import os
import re
import time
import random
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

# ============================== CONFIG ==============================

EXCEL_PATH = "qcm_final.xlsx"
SHEET_NAME = "Feuil2"

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "sk-or-v1-1533e7d6a29707afd")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

# nom complet OpenRouter -> nom court utilisé dans les noms de colonnes
MODEL_MAP = {
    "google/gemma-4-26b-a4b-it": "gemma4_26b",
    "qwen/qwen3.6-27b": "qwen3_6_27b",
    "qwen/qwen3.5-9b": "qwen3_5_9b",
    "cohere/command-r-08-2024": "commandr",
    "google/gemma-4-31b-it": "gemma4_31b",
    "openai/gpt-oss-20b": "gptoss_20b",
    "openai/gpt-5": "gpt5",
    "anthropic/claude-sonnet-5": "claude_sonnet5",
    "google/gemini-3.6-flash": "gemini3_6_flash",
    "x-ai/grok-4.20": "grok4_20",
    "mistralai/mistral-small-3.2-24b-instruct": "mistral_24b",
    "mistralai/mistral-saba": "mistral_saba",
}

# --- structure du QCM ---
OPTION_COLS = ["response_A", "response_B", "response_C", "response_D"]  # celles absentes sont ignorées
ANSWER_COL = "final_answer"   # vérité terrain utilisée pour le score (pas original_key)

# --- concurrence / robustesse ---
MAX_WORKERS = 10           # nb d'appels API en parallèle
MAX_RETRIES = 4            # nb de tentatives par appel en cas d'erreur transitoire
RETRY_BACKOFF_SECONDS = 5  # backoff de base (multiplié par le n° de tentative)
REQUEST_TIMEOUT = 90       # timeout HTTP en secondes
SAVE_EVERY = 25            # sauvegarder le classeur toutes les N lignes terminées
JITTER_BEFORE_CALL = 0.15  # petit décalage aléatoire avant chaque appel

# ============================== PROMPT ==============================

def get_option_cols(df) -> list:
    """Colonnes d'options réellement présentes dans le classeur, dans l'ordre."""
    return [c for c in OPTION_COLS if c in df.columns]

def build_prompt(row, option_cols) -> tuple:
    """
    Construit le prompt et retourne (prompt, lettres_valides).
    Les lettres sont déduites du nom des colonnes : response_A -> A, etc.
    Une option vide (NaN) est simplement ignorée.
    """
    lines, letters = [], []
    for col in option_cols:
        value = row[col]
        if pd.isna(value) or str(value).strip() == "":
            continue
        letter = col.split("_")[-1].upper()
        letters.append(letter)
        lines.append(f"{letter}: {value}")

    allowed = " ولا ".join(letters)
    prompt = (
        f"{row['input']}\n\n"
        + "\n".join(lines)
        + f"\n\nجاوب بحرف واحد فقط: {allowed}. بلا شرح، بلا علامات، بلا كلام زايد."
    )
    return prompt, letters

# ============================== PARSING ==============================

def parse_answer(raw_text, letters):
    """
    Extrait la lettre de réponse du texte brut. Retourne la lettre ou None.
    Prend le DERNIER match (au cas où le modèle recopie l'énoncé avant sa réponse).
    """
    if not raw_text:
        return None
    text = str(raw_text).strip()
    if text in letters:
        return text
    charset = "".join(letters)
    matches = re.findall(rf'(?<![A-Za-z])([{charset}])(?![A-Za-z])', text)
    if matches:
        return matches[-1]
    return None

# ============================== APPEL API ==============================

# mode "reasoning" par modèle : "off" -> désactivé, "on" -> activé, "omit" -> champ absent
REASONING_MODE = {}
_reasoning_lock = threading.Lock()
_thread_local = threading.local()

REASONING_MANDATORY_PAT = re.compile(r"reasoning is mandatory", re.IGNORECASE)

def get_session() -> requests.Session:
    """Une session HTTP par thread (connexions réutilisées, thread-safe)."""
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        adapter = requests.adapters.HTTPAdapter(pool_connections=MAX_WORKERS,
                                                pool_maxsize=MAX_WORKERS)
        s.mount("https://", adapter)
        _thread_local.session = s
    return s

def get_mode(model: str) -> str:
    with _reasoning_lock:
        return REASONING_MODE.setdefault(model, "off")

def set_mode(model: str, mode: str):
    with _reasoning_lock:
        if REASONING_MODE.get(model) != mode:
            REASONING_MODE[model] = mode
            print(f"  >>> [{model}] bascule du paramètre reasoning en '{mode}'")

def build_payload(model: str, prompt: str, mode: str) -> dict:
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
    }
    if mode == "off":
        payload["reasoning"] = {"enabled": False}
    elif mode == "on":
        payload["reasoning"] = {"enabled": True}
    # mode == "omit" -> on n'envoie rien
    return payload

def call_openrouter(model: str, prompt: str):
    """
    Appelle OpenRouter avec retries + négociation automatique du champ reasoning.
    Retourne (contenu_texte, erreur) : un seul des deux est non-None.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    session = get_session()
    last_error = None
    tried_modes = set()

    for attempt in range(1, MAX_RETRIES + 1):
        mode = get_mode(model)
        tried_modes.add(mode)
        try:
            resp = session.post(API_URL, headers=headers,
                                json=build_payload(model, prompt, mode),
                                timeout=REQUEST_TIMEOUT)
        except requests.exceptions.RequestException as e:
            last_error = f"request_exception: {e}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
            continue

        if resp.status_code == 200:
            try:
                data = resp.json()
                msg = data["choices"][0]["message"]
                content = msg.get("content")
                if not content:  # certains modèles renvoient tout dans "reasoning"
                    content = msg.get("reasoning") or ""
                return content, None
            except Exception as e:
                return None, f"unexpected_response_format: {e} | raw={resp.text[:300]}"

        body = resp.text[:500]

        # --- reasoning obligatoire, ou champ reasoning problématique ---
        if resp.status_code in (400, 404, 422) and REASONING_MANDATORY_PAT.search(body):
            set_mode(model, "on")
            continue  # on retente tout de suite, sans backoff
        if resp.status_code in (400, 404, 422) and "reasoning" in body.lower():
            if "omit" not in tried_modes:
                set_mode(model, "omit")
                continue

        if resp.status_code == 429:
            last_error = f"rate_limited (429): {body[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt * 2 + random.uniform(0, 2))
        elif 500 <= resp.status_code < 600:
            last_error = f"server_error ({resp.status_code}): {body[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
        else:
            return None, f"http_{resp.status_code}: {body[:300]}"

    return None, last_error

# ============================== I/O EXCEL ==============================

def load_all_sheets(path: str) -> dict:
    if not Path(path).exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")
    return pd.read_excel(path, sheet_name=None)

def save_all_sheets(sheets: dict, path: str):
    """Écrit dans un fichier temporaire puis remplace l'original (évite un fichier
    corrompu si le process est tué pendant l'écriture)."""
    p = Path(path)
    tmp_path = p.with_name(p.stem + ".tmp" + p.suffix)
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
        for name, sheet_df in sheets.items():
            sheet_df.to_excel(writer, sheet_name=name, index=False)
    tmp_path.replace(path)

# ============================== BOUCLE PRINCIPALE ==============================

# Variables globales : si une sauvegarde échoue et que le run est interrompu,
# "sheets" et "df" restent accessibles depuis le notebook -> sauvegarde manuelle :
#   save_all_sheets(sheets, EXCEL_PATH)
sheets = None
df = None

def _work(model_full, idx, prompt, letters):
    """Exécuté dans un thread : ne touche jamais au dataframe."""
    if JITTER_BEFORE_CALL:
        time.sleep(random.uniform(0, JITTER_BEFORE_CALL))
    raw_text, error = call_openrouter(model_full, prompt)
    parsed = parse_answer(raw_text, letters) if error is None else None
    return idx, raw_text, error, parsed

def run_evaluation():
    global sheets, df
    sheets = load_all_sheets(EXCEL_PATH)
    df = sheets[SHEET_NAME]
    option_cols = get_option_cols(df)
    if not option_cols:
        raise ValueError(f"Aucune colonne d'options trouvée parmi {OPTION_COLS}")
    print(f"Colonnes d'options utilisées : {option_cols}")

    for model_full, model_short in MODEL_MAP.items():
        col_raw = f"response_{model_short}_raw"
        col_final = f"response_{model_short}"

        if col_raw not in df.columns:
            df[col_raw] = pd.NA
        if col_final not in df.columns:
            df[col_final] = pd.NA

        todo = [i for i in df.index if pd.isna(df.at[i, col_final])]
        print(f"\n=== {model_full} ({model_short}) — {len(todo)} ligne(s) restante(s) ===")
        if not todo:
            continue

        done = 0
        done_since_save = 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {}
            for idx in todo:
                try:
                    prompt, letters = build_prompt(df.loc[idx], option_cols)
                except Exception as e:
                    df.at[idx, col_raw] = f"[PROMPT_ERROR] {e}"
                    print(f"  [{idx}] impossible de construire le prompt : {e}")
                    continue
                futures[pool.submit(_work, model_full, idx, prompt, letters)] = (idx, letters)

            for fut in as_completed(futures):
                idx, letters = futures[fut]
                item_id = df.at[idx, "item_id"] if "item_id" in df.columns else idx
                try:
                    idx, raw_text, error, parsed = fut.result()
                    if error is not None:
                        df.at[idx, col_raw] = f"[ERROR] {error}"
                        # col_final reste NA -> sera retenté au prochain run
                        print(f"  [{item_id}] ERREUR API: {error}")
                    else:
                        df.at[idx, col_raw] = raw_text
                        if parsed is None:
                            df.at[idx, col_final] = "UNPARSED"
                            print(f"  [{item_id}] parsing impossible, brut conservé: {str(raw_text)[:120]!r}")
                        else:
                            df.at[idx, col_final] = parsed
                except Exception as e:
                    df.at[idx, col_raw] = f"[UNEXPECTED_ERROR] {e}"
                    print(f"  [{item_id}] ERREUR INATTENDUE: {e}")

                done += 1
                done_since_save += 1
                if done % 50 == 0:
                    print(f"  ... {done}/{len(futures)}")
                if done_since_save >= SAVE_EVERY:
                    try:
                        sheets[SHEET_NAME] = df
                        save_all_sheets(sheets, EXCEL_PATH)
                    except Exception as e:
                        print(f"  (sauvegarde échouée, on continue quand même: {e})")
                    done_since_save = 0

        # sauvegarde de fin de modèle
        try:
            sheets[SHEET_NAME] = df
            save_all_sheets(sheets, EXCEL_PATH)
        except Exception as e:
            print(f"Sauvegarde finale échouée pour {model_short}: {e}")

    print("\nTerminé.")
    print_scores()

# ============================== SCORES ==============================

def print_scores():
    """Récapitulatif : exactitude par modèle (sur les lignes effectivement parsées)."""
    if df is None or ANSWER_COL not in df.columns:
        return
    valid_letters = ["A", "B", "C", "D"]
    gold = df[ANSWER_COL].astype(str).str.strip().str.upper()
    print("\n--- Exactitude (vs %s) ---" % ANSWER_COL)
    for model_short in MODEL_MAP.values():
        col = f"response_{model_short}"
        if col not in df.columns:
            continue
        pred = df[col].astype(str).str.strip().str.upper()
        answered = pred.isin(valid_letters)
        n = int(answered.sum())
        n_unparsed = int((pred == "UNPARSED").sum())
        n_missing = int(df[col].isna().sum())
        if n == 0:
            print(f"{model_short:>18} : aucune réponse exploitable "
                  f"(unparsed={n_unparsed}, manquants={n_missing})")
            continue
        acc = float((pred[answered] == gold[answered]).mean())
        print(f"{model_short:>18} : {acc:.1%} sur {n} items "
              f"(unparsed={n_unparsed}, manquants={n_missing})")


try:
    run_evaluation()
except KeyboardInterrupt:
    print("\nInterrompu manuellement — les résultats déjà obtenus sont sauvegardés. "
          "Ré-exécute la cellule pour reprendre là où tu t'es arrêté.")


Colonnes d'options utilisées : ['response_A', 'response_B', 'response_C', 'response_D']

=== google/gemma-4-26b-a4b-it (gemma4_26b) — 577 ligne(s) restante(s) ===
  ... 50/577
  ... 100/577
  ... 150/577
  ... 200/577
  ... 250/577
  ... 300/577
  ... 350/577
  ... 400/577
  ... 450/577
  ... 500/577
  ... 550/577

=== qwen/qwen3.6-27b (qwen3_6_27b) — 577 ligne(s) restante(s) ===
  ... 50/577
  ... 100/577
  ... 150/577
  ... 200/577
  ... 250/577
  ... 300/577
  ... 350/577
  ... 400/577
  ... 450/577
  ... 500/577
  ... 550/577

=== qwen/qwen3.5-9b (qwen3_5_9b) — 577 ligne(s) restante(s) ===
  ... 50/577
  ... 100/577
  ... 150/577
  ... 200/577
  ... 250/577
  ... 300/577
  ... 350/577
  [QCM_412] ERREUR API: unexpected_response_format: 'choices' | raw=
         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

  

In [4]:
"""
Évaluation des modèles OpenRouter sur le classeur QCM (4 options A/B/C/D)
--------------------------------------------------------------------------------
Colonnes attendues dans la feuille :
  item_id, item_type, Axe, Difficulty, intent_group, intent_category,
  input, response_A, response_B, response_C, response_D, original_key, final_answer

Le script ajoute pour chaque modèle :
  response_<nom_court>_raw    -> texte brut renvoyé par le modèle
  response_<nom_court>        -> lettre parsée ("A".."D" ou "UNPARSED")

Caractéristiques :
- Appels concurrents (MAX_WORKERS, 10 par défaut).
- Gestion automatique du champ "reasoning" : on tente d'abord désactivé ; si
  l'endpoint répond « Reasoning is mandatory... », le modèle bascule en reasoning
  activé (puis, en dernier recours, le champ est omis). Le mode qui marche est
  mémorisé pour tous les appels suivants de ce modèle.
- Sauvegarde régulière + reprise : les lignes déjà remplies sont sautées, donc il
  suffit de ré-exécuter la cellule après un crash ou un Ctrl-C.
- Aucune erreur réseau/API/parsing n'arrête le run.
- Récapitulatif d'exactitude en fin de run (colonne final_answer).

À faire avant de lancer :
1. Mets ta clé dans OPENROUTER_API_KEY (variable d'environnement ou en dur).
2. Vérifie EXCEL_PATH et SHEET_NAME.
"""

import os
import re
import time
import random
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

# ============================== CONFIG ==============================

EXCEL_PATH = "qcm_final.xlsx"
SHEET_NAME = "Feuil2"

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "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")
API_URL = "https://openrouter.ai/api/v1/chat/completions"



# nom complet OpenRouter -> nom court utilisé dans les noms de colonnes
MODEL_MAP = {
    "google/gemma-4-26b-a4b-it": "gemma4_26b",
    "qwen/qwen3.6-27b": "qwen3_6_27b",
    "qwen/qwen3.5-9b": "qwen3_5_9b",
    "cohere/command-r-08-2024": "commandr",
    "google/gemma-4-31b-it": "gemma4_31b",
    "openai/gpt-oss-20b": "gptoss_20b",
    "openai/gpt-5": "gpt5",
    "anthropic/claude-sonnet-5": "claude_sonnet5",
    "google/gemini-3.6-flash": "gemini3_6_flash",
    "x-ai/grok-4.20": "grok4_20",
    "mistralai/mistral-small-3.2-24b-instruct": "mistral_24b",
    "mistralai/mistral-saba": "mistral_saba",
}

# --- structure du QCM ---
OPTION_COLS = ["response_A", "response_B", "response_C", "response_D"]  # celles absentes sont ignorées
ANSWER_COL = "final_answer"   # vérité terrain utilisée pour le score (pas original_key)

# --- concurrence / robustesse ---
MAX_WORKERS = 10           # nb d'appels API en parallèle
MAX_RETRIES = 4            # nb de tentatives par appel en cas d'erreur transitoire
RETRY_BACKOFF_SECONDS = 5  # backoff de base (multiplié par le n° de tentative)
REQUEST_TIMEOUT = 90       # timeout HTTP en secondes
SAVE_EVERY = 25            # sauvegarder le classeur toutes les N lignes terminées
JITTER_BEFORE_CALL = 0.15  # petit décalage aléatoire avant chaque appel

# ============================== PROMPT ==============================

def get_option_cols(df) -> list:
    """Colonnes d'options réellement présentes dans le classeur, dans l'ordre."""
    return [c for c in OPTION_COLS if c in df.columns]

def build_prompt(row, option_cols) -> tuple:
    """
    Construit le prompt et retourne (prompt, lettres_valides).
    Les lettres sont déduites du nom des colonnes : response_A -> A, etc.
    Une option vide (NaN) est simplement ignorée.
    """
    lines, letters = [], []
    for col in option_cols:
        value = row[col]
        if pd.isna(value) or str(value).strip() == "":
            continue
        letter = col.split("_")[-1].upper()
        letters.append(letter)
        lines.append(f"{letter}: {value}")

    allowed = " ولا ".join(letters)
    prompt = (
        f"{row['input']}\n\n"
        + "\n".join(lines)
        + f"\n\nجاوب بحرف واحد فقط: {allowed}. بلا شرح، بلا علامات، بلا كلام زايد."
    )
    return prompt, letters

# ============================== PARSING ==============================

def parse_answer(raw_text, letters):
    """
    Extrait la lettre de réponse du texte brut. Retourne la lettre ou None.
    Prend le DERNIER match (au cas où le modèle recopie l'énoncé avant sa réponse).
    """
    if not raw_text:
        return None
    text = str(raw_text).strip()
    if text in letters:
        return text
    charset = "".join(letters)
    matches = re.findall(rf'(?<![A-Za-z])([{charset}])(?![A-Za-z])', text)
    if matches:
        return matches[-1]
    return None

# ============================== APPEL API ==============================

# mode "reasoning" par modèle : "off" -> désactivé, "on" -> activé, "omit" -> champ absent
REASONING_MODE = {}
_reasoning_lock = threading.Lock()
_thread_local = threading.local()

REASONING_MANDATORY_PAT = re.compile(r"reasoning is mandatory", re.IGNORECASE)

def get_session() -> requests.Session:
    """Une session HTTP par thread (connexions réutilisées, thread-safe)."""
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        adapter = requests.adapters.HTTPAdapter(pool_connections=MAX_WORKERS,
                                                pool_maxsize=MAX_WORKERS)
        s.mount("https://", adapter)
        _thread_local.session = s
    return s

def get_mode(model: str) -> str:
    with _reasoning_lock:
        return REASONING_MODE.setdefault(model, "off")

def set_mode(model: str, mode: str):
    with _reasoning_lock:
        if REASONING_MODE.get(model) != mode:
            REASONING_MODE[model] = mode
            print(f"  >>> [{model}] bascule du paramètre reasoning en '{mode}'")

def build_payload(model: str, prompt: str, mode: str) -> dict:
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
    }
    if mode == "off":
        payload["reasoning"] = {"enabled": False}
    elif mode == "on":
        payload["reasoning"] = {"enabled": True}
    # mode == "omit" -> on n'envoie rien
    return payload

def call_openrouter(model: str, prompt: str):
    """
    Appelle OpenRouter avec retries + négociation automatique du champ reasoning.
    Retourne (contenu_texte, erreur) : un seul des deux est non-None.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    session = get_session()
    last_error = None
    tried_modes = set()

    for attempt in range(1, MAX_RETRIES + 1):
        mode = get_mode(model)
        tried_modes.add(mode)
        try:
            resp = session.post(API_URL, headers=headers,
                                json=build_payload(model, prompt, mode),
                                timeout=REQUEST_TIMEOUT)
        except requests.exceptions.RequestException as e:
            last_error = f"request_exception: {e}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
            continue

        if resp.status_code == 200:
            try:
                data = resp.json()
                msg = data["choices"][0]["message"]
                content = msg.get("content")
                if not content:  # certains modèles renvoient tout dans "reasoning"
                    content = msg.get("reasoning") or ""
                return content, None
            except Exception as e:
                return None, f"unexpected_response_format: {e} | raw={resp.text[:300]}"

        body = resp.text[:500]

        # --- reasoning obligatoire, ou champ reasoning problématique ---
        if resp.status_code in (400, 404, 422) and REASONING_MANDATORY_PAT.search(body):
            set_mode(model, "on")
            continue  # on retente tout de suite, sans backoff
        if resp.status_code in (400, 404, 422) and "reasoning" in body.lower():
            if "omit" not in tried_modes:
                set_mode(model, "omit")
                continue

        if resp.status_code == 429:
            last_error = f"rate_limited (429): {body[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt * 2 + random.uniform(0, 2))
        elif 500 <= resp.status_code < 600:
            last_error = f"server_error ({resp.status_code}): {body[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
        else:
            return None, f"http_{resp.status_code}: {body[:300]}"

    return None, last_error

# ============================== I/O EXCEL ==============================

def load_all_sheets(path: str) -> dict:
    if not Path(path).exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")
    return pd.read_excel(path, sheet_name=None)

def save_all_sheets(sheets: dict, path: str):
    """Écrit dans un fichier temporaire puis remplace l'original (évite un fichier
    corrompu si le process est tué pendant l'écriture)."""
    p = Path(path)
    tmp_path = p.with_name(p.stem + ".tmp" + p.suffix)
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
        for name, sheet_df in sheets.items():
            sheet_df.to_excel(writer, sheet_name=name, index=False)
    tmp_path.replace(path)

# ============================== BOUCLE PRINCIPALE ==============================

# Variables globales : si une sauvegarde échoue et que le run est interrompu,
# "sheets" et "df" restent accessibles depuis le notebook -> sauvegarde manuelle :
#   save_all_sheets(sheets, EXCEL_PATH)
sheets = None
df = None

def _work(model_full, idx, prompt, letters):
    """Exécuté dans un thread : ne touche jamais au dataframe."""
    if JITTER_BEFORE_CALL:
        time.sleep(random.uniform(0, JITTER_BEFORE_CALL))
    raw_text, error = call_openrouter(model_full, prompt)
    parsed = parse_answer(raw_text, letters) if error is None else None
    return idx, raw_text, error, parsed

def run_evaluation():
    global sheets, df
    sheets = load_all_sheets(EXCEL_PATH)
    df = sheets[SHEET_NAME]
    option_cols = get_option_cols(df)
    if not option_cols:
        raise ValueError(f"Aucune colonne d'options trouvée parmi {OPTION_COLS}")
    print(f"Colonnes d'options utilisées : {option_cols}")

    for model_full, model_short in MODEL_MAP.items():
        col_raw = f"response_{model_short}_raw"
        col_final = f"response_{model_short}"

        if col_raw not in df.columns:
            df[col_raw] = pd.NA
        if col_final not in df.columns:
            df[col_final] = pd.NA

        todo = [i for i in df.index if pd.isna(df.at[i, col_final])]
        print(f"\n=== {model_full} ({model_short}) — {len(todo)} ligne(s) restante(s) ===")
        if not todo:
            continue

        done = 0
        done_since_save = 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {}
            for idx in todo:
                try:
                    prompt, letters = build_prompt(df.loc[idx], option_cols)
                except Exception as e:
                    df.at[idx, col_raw] = f"[PROMPT_ERROR] {e}"
                    print(f"  [{idx}] impossible de construire le prompt : {e}")
                    continue
                futures[pool.submit(_work, model_full, idx, prompt, letters)] = (idx, letters)

            for fut in as_completed(futures):
                idx, letters = futures[fut]
                item_id = df.at[idx, "item_id"] if "item_id" in df.columns else idx
                try:
                    idx, raw_text, error, parsed = fut.result()
                    if error is not None:
                        df.at[idx, col_raw] = f"[ERROR] {error}"
                        # col_final reste NA -> sera retenté au prochain run
                        print(f"  [{item_id}] ERREUR API: {error}")
                    else:
                        df.at[idx, col_raw] = raw_text
                        if parsed is None:
                            df.at[idx, col_final] = "UNPARSED"
                            print(f"  [{item_id}] parsing impossible, brut conservé: {str(raw_text)[:120]!r}")
                        else:
                            df.at[idx, col_final] = parsed
                except Exception as e:
                    df.at[idx, col_raw] = f"[UNEXPECTED_ERROR] {e}"
                    print(f"  [{item_id}] ERREUR INATTENDUE: {e}")

                done += 1
                done_since_save += 1
                if done % 50 == 0:
                    print(f"  ... {done}/{len(futures)}")
                if done_since_save >= SAVE_EVERY:
                    try:
                        sheets[SHEET_NAME] = df
                        save_all_sheets(sheets, EXCEL_PATH)
                    except Exception as e:
                        print(f"  (sauvegarde échouée, on continue quand même: {e})")
                    done_since_save = 0

        # sauvegarde de fin de modèle
        try:
            sheets[SHEET_NAME] = df
            save_all_sheets(sheets, EXCEL_PATH)
        except Exception as e:
            print(f"Sauvegarde finale échouée pour {model_short}: {e}")

    print("\nTerminé.")
    print_scores()

# ============================== SCORES ==============================

def print_scores():
    """Récapitulatif : exactitude par modèle (sur les lignes effectivement parsées)."""
    if df is None or ANSWER_COL not in df.columns:
        return
    valid_letters = ["A", "B", "C", "D"]
    gold = df[ANSWER_COL].astype(str).str.strip().str.upper()
    print("\n--- Exactitude (vs %s) ---" % ANSWER_COL)
    for model_short in MODEL_MAP.values():
        col = f"response_{model_short}"
        if col not in df.columns:
            continue
        pred = df[col].astype(str).str.strip().str.upper()
        answered = pred.isin(valid_letters)
        n = int(answered.sum())
        n_unparsed = int((pred == "UNPARSED").sum())
        n_missing = int(df[col].isna().sum())
        if n == 0:
            print(f"{model_short:>18} : aucune réponse exploitable "
                  f"(unparsed={n_unparsed}, manquants={n_missing})")
            continue
        acc = float((pred[answered] == gold[answered]).mean())
        print(f"{model_short:>18} : {acc:.1%} sur {n} items "
              f"(unparsed={n_unparsed}, manquants={n_missing})")


try:
    run_evaluation()
except KeyboardInterrupt:
    print("\nInterrompu manuellement — les résultats déjà obtenus sont sauvegardés. "
          "Ré-exécute la cellule pour reprendre là où tu t'es arrêté.")


Colonnes d'options utilisées : ['response_A', 'response_B', 'response_C', 'response_D']

=== google/gemma-4-26b-a4b-it (gemma4_26b) — 0 ligne(s) restante(s) ===

=== qwen/qwen3.6-27b (qwen3_6_27b) — 0 ligne(s) restante(s) ===

=== qwen/qwen3.5-9b (qwen3_5_9b) — 3 ligne(s) restante(s) ===

=== cohere/command-r-08-2024 (commandr) — 0 ligne(s) restante(s) ===

=== google/gemma-4-31b-it (gemma4_31b) — 0 ligne(s) restante(s) ===

=== openai/gpt-oss-20b (gptoss_20b) — 0 ligne(s) restante(s) ===

=== openai/gpt-5 (gpt5) — 0 ligne(s) restante(s) ===

=== anthropic/claude-sonnet-5 (claude_sonnet5) — 0 ligne(s) restante(s) ===

=== google/gemini-3.6-flash (gemini3_6_flash) — 0 ligne(s) restante(s) ===

=== x-ai/grok-4.20 (grok4_20) — 0 ligne(s) restante(s) ===

=== mistralai/mistral-small-3.2-24b-instruct (mistral_24b) — 10 ligne(s) restante(s) ===
  [QCM_206] parsing impossible, brut conservé: 'رئيس الوزراء'

=== mistralai/mistral-saba (mistral_saba) — 0 ligne(s) restante(s) ===

Terminé.

--- 